In [1]:
import os
import random
import numpy as np
import pandas as pd

SEED = 24
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# BUGFIX: Do NOT force pure-python protobuf; it breaks TF in this environment
# (it caused: AttributeError: 'MessageFactory' object has no attribute 'GetPrototype')
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", None)

random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras.models as M
import tensorflow.keras.backend as K
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler

tf.random.set_seed(SEED)



2026-05-12 14:09:33.755988: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778594973.766341      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778594973.769509      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
EPOCHS = 5
NUM_IMAGES = 40  # minimum depth that remains valid through all conv/pool layers
BATCH_SIZE = 8
FOLDS = 5
IMAGE_DIM = (NUM_IMAGES, 60, 60)

COMP_DIR = "../input/osic-pulmonary-fibrosis-progression/"
TRAIN_PATH = "../input/osic-pulmonary-fibrosis-progression/train"
TEST_PATH = "../input/osic-pulmonary-fibrosis-progression/test"
SUB_PATH = "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"



In [3]:
comp_dir = "../input/osic-pulmonary-fibrosis-progression"

train_data = pd.read_csv(os.path.join(comp_dir, "train.csv"))
sub = pd.read_csv(os.path.join(comp_dir, "sample_submission.csv"))
test_data = pd.read_csv(os.path.join(comp_dir, "test.csv"))

train_data = train_data.drop_duplicates(keep=False, subset=["Patient", "Weeks"])

train_data.head(), test_data.head(), sub.head()



(                     Patient  Weeks   FVC    Percent  Age   Sex SmokingStatus
 0  ID00133637202223847701934     -2  3195  92.856312   83  Male  Never smoked
 1  ID00133637202223847701934      2  3203  93.088817   83  Male  Never smoked
 2  ID00133637202223847701934      4  3097  90.008138   83  Male  Never smoked
 3  ID00133637202223847701934      6  3171  92.158800   83  Male  Never smoked
 4  ID00133637202223847701934      8  3115  90.531272   83  Male  Never smoked,
                      Patient  Weeks   FVC    Percent  Age     Sex  \
 0  ID00014637202177757139317      0  3807  90.076661   56    Male   
 1  ID00019637202178323708467     13  2100  92.858722   83  Female   
 2  ID00047637202184938901501      2  3313  89.929425   68    Male   
 3  ID00082637202201836229724     19  2918  99.794802   49  Female   
 4  ID00126637202218610655908     18  2375  59.375000   78    Male   
 
       SmokingStatus  
 0         Ex-smoker  
 1         Ex-smoker  
 2         Ex-smoker  
 3  Current

In [4]:
train_data_u = train_data.copy()
train_data_u = train_data_u.drop_duplicates(subset=["Patient"])
train_data_u = train_data_u.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
train_data_u["Typical_FVC"] = (
    train_data_u.Base_FVC.values / train_data_u.Base_Percent.values
) * 100
train_data = train_data.merge(
    train_data_u.drop(["Age", "Sex", "SmokingStatus"], axis=1), on="Patient", how="left"
)

train_data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Base_Week,Base_FVC,Base_Percent,Typical_FVC
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked,-2,3195,92.856312,3440.8
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked,-2,3195,92.856312,3440.8
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked,-2,3195,92.856312,3440.8
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked,-2,3195,92.856312,3440.8
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked,-2,3195,92.856312,3440.8


In [5]:
sub["Patient"] = sub["Patient_Week"].str.split("_").str[0]
sub["Weeks"] = sub["Patient_Week"].str.split("_").str[-1].astype(int)

sub = sub.drop(["Confidence"], axis=1)
sub = sub[["Patient", "Weeks", "Patient_Week"]]
sub.head()



,Patient,Weeks,Patient_Week
0,ID00126637202218610655908,-3,ID00126637202218610655908_-3
1,ID00126637202218610655908,-2,ID00126637202218610655908_-2
2,ID00126637202218610655908,-1,ID00126637202218610655908_-1
3,ID00126637202218610655908,0,ID00126637202218610655908_0
4,ID00126637202218610655908,1,ID00126637202218610655908_1


In [6]:
test_data = test_data.rename(
    columns={"Weeks": "Base_Week", "FVC": "Base_FVC", "Percent": "Base_Percent"}
)
test_data["Typical_FVC"] = (
    test_data.Base_FVC.values / test_data.Base_Percent.values
) * 100

sub = sub.merge(test_data, how="left", on="Patient")
sub.head()



,Patient,Weeks,Patient_Week,Base_Week,Base_FVC,Base_Percent,Age,Sex,SmokingStatus,Typical_FVC
0,ID00126637202218610655908,-3,ID00126637202218610655908_-3,18,2375,59.375,78,Male,Ex-smoker,4000.0
1,ID00126637202218610655908,-2,ID00126637202218610655908_-2,18,2375,59.375,78,Male,Ex-smoker,4000.0
2,ID00126637202218610655908,-1,ID00126637202218610655908_-1,18,2375,59.375,78,Male,Ex-smoker,4000.0
3,ID00126637202218610655908,0,ID00126637202218610655908_0,18,2375,59.375,78,Male,Ex-smoker,4000.0
4,ID00126637202218610655908,1,ID00126637202218610655908_1,18,2375,59.375,78,Male,Ex-smoker,4000.0


In [7]:
train_data["Type"] = "train"
sub["Type"] = "test"

data = pd.concat([train_data, sub], axis=0, ignore_index=True)

data.columns



Index(['Patient', 'Weeks', 'FVC', 'Percent', 'Age', 'Sex', 'SmokingStatus',
       'Base_Week', 'Base_FVC', 'Base_Percent', 'Typical_FVC', 'Type',
       'Patient_Week'],
      dtype='object')

In [8]:
prediction_col = ["FVC"]
Continuos_cols = [
    "Weeks",
    "Base_Week",
    "Base_FVC",
    "Typical_FVC",
    "Age",
    "Percent",
    "Base_Percent",
]

Categorical_cols = ["Sex", "SmokingStatus"]



In [9]:
for c in Continuos_cols:
    if c not in data.columns:
        data[c] = np.nan
data[Continuos_cols] = data[Continuos_cols].fillna(
    data[Continuos_cols].median(numeric_only=True)
)

scaler = MinMaxScaler()
data[Continuos_cols] = scaler.fit_transform(data[Continuos_cols])

data["Sex"] = data["Sex"].fillna("Male")
data["SmokingStatus"] = data["SmokingStatus"].fillna("Never smoked")

data.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,Base_Week,Base_FVC,Base_Percent,Typical_FVC,Type,Patient_Week
0,ID00133637202223847701934,0.021739,3195.0,0.514846,0.871795,Male,Never smoked,0.035714,0.404903,0.451429,0.554072,train,NaN
1,ID00133637202223847701934,0.050725,3203.0,0.516717,0.871795,Male,Never smoked,0.035714,0.404903,0.451429,0.554072,train,NaN
2,ID00133637202223847701934,0.065217,3097.0,0.491926,0.871795,Male,Never smoked,0.035714,0.404903,0.451429,0.554072,train,NaN
3,ID00133637202223847701934,0.079710,3171.0,0.509233,0.871795,Male,Never smoked,0.035714,0.404903,0.451429,0.554072,train,NaN
4,ID00133637202223847701934,0.094203,3115.0,0.496136,0.871795,Male,Never smoked,0.035714,0.404903,0.451429,0.554072,train,NaN


In [10]:
data["Sex"] = data["Sex"].map({"Male": 0, "Female": 1}).fillna(0).astype(np.float32)

smoke_map = {"Ex-smoker": 0, "Never smoked": 1, "Currently smokes": 2}
data["SmokingStatus"] = (
    data["SmokingStatus"].map(smoke_map).fillna(1).astype(np.float32)
)

data[["Sex", "SmokingStatus"]].head()



,Sex,SmokingStatus
0,0.0,1.0
1,0.0,1.0
2,0.0,1.0
3,0.0,1.0
4,0.0,1.0


In [11]:
x_cols = ["Weeks", "Base_Week", "Base_FVC", "Sex", "Age"]

train_mask = data["Type"].values == "train"
test_mask = ~train_mask

x_train = data.loc[train_mask, x_cols].to_numpy(dtype=np.float32, copy=True)
y_train = data.loc[train_mask, prediction_col].to_numpy(dtype=np.float32, copy=True)
x_test = data.loc[test_mask, x_cols].to_numpy(dtype=np.float32, copy=True)

patient_ids_train = data.loc[train_mask, "Patient"].to_numpy(copy=True)
patient_ids_test = data.loc[test_mask, "Patient"].to_numpy(copy=True)

x_train.shape, y_train.shape, x_test.shape, len(patient_ids_test)



((1380, 5), (1380, 1), (1908, 5), 1908)

In [12]:
_ZERO_VOL_CACHE = np.zeros((*IMAGE_DIM, 1), dtype=np.float32)
_ZERO_VOL_TF_1 = tf.constant(
    _ZERO_VOL_CACHE[None, ...], dtype=tf.float32
)  # (1,D,H,W,1)



2026-05-12 14:09:39.557569: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [13]:
C1, C2 = tf.constant(70, dtype="float32"), tf.constant(1000, dtype="float32")


def score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    sigma = y_pred[:, 2] - y_pred[:, 0]
    fvc_pred = y_pred[:, 1]
    sigma_clip = tf.maximum(sigma, C1)
    delta = tf.abs(y_true[:, 0] - fvc_pred)
    delta = tf.minimum(delta, C2)
    sq2 = tf.sqrt(tf.cast(2.0, dtype=tf.float32))
    metric = (delta / sigma_clip) * sq2 + tf.math.log(sigma_clip * sq2)
    return K.mean(metric)


def qloss(y_true, y_pred):
    qs = [0.2, 0.50, 0.8]
    q = tf.constant(np.array([qs]), dtype=tf.float32)
    e = y_true - y_pred
    v = tf.maximum(q * e, (q - 1) * e)
    return K.mean(v)


def mloss(_lambda):
    def loss(y_true, y_pred):
        return _lambda * qloss(y_true, y_pred) + (1 - _lambda) * score(y_true, y_pred)

    return loss




In [14]:
def build_model():
    inp2 = L.Input(shape=(*IMAGE_DIM, 1), name="conv_input")

    x = L.Conv3D(32, 3, activation="relu", data_format="channels_last")(inp2)
    x = L.Conv3D(64, 3, activation="relu", data_format="channels_last")(x)
    x = L.BatchNormalization()(x)
    x1 = L.MaxPooling3D()(x)

    x = L.Conv3D(64, 3, activation="relu", padding="same", data_format="channels_last")(
        x1
    )
    x = L.Conv3D(64, 3, activation="relu", padding="same", data_format="channels_last")(
        x
    )
    x = L.add([x, x1])

    x = L.Conv3D(128, 5, activation="relu", data_format="channels_last")(x)
    x = L.Conv3D(128, 5, activation="relu", data_format="channels_last")(x)
    x = L.BatchNormalization()(x)
    x2 = L.MaxPooling3D()(x)

    x = L.Conv3D(
        128, 5, activation="relu", padding="same", data_format="channels_last"
    )(x2)
    x = L.Conv3D(
        128, 5, activation="relu", padding="same", data_format="channels_last"
    )(x)
    x = L.add([x, x2])

    x = L.BatchNormalization()(x)
    x = L.MaxPooling3D()(x)
    x = L.Dropout(0.5)(x)

    op1 = L.Flatten(name="flatten")(x)

    inp = L.Input((5,), name="input_d")
    d = L.Dense(128, activation="relu", name="dense")(inp)
    d1 = L.Dense(128, activation="relu", name="dense_1")(d)
    op2 = L.Dense(64, activation="relu", name="dense_2")(d1)

    x = L.Concatenate(name="concatenate")([op1, op2])

    o1 = L.Dense(3, activation="linear", name="dense_f1")(x)
    o2 = L.Dense(3, activation="relu", name="dense_f2")(x)

    pred1 = L.Lambda(lambda z: (z[0] + tf.cumsum(z[1], axis=1)), name="output")(
        [o1, o2]
    )

    model = M.Model(inputs=[inp2, inp], outputs=pred1)

    model.compile(
        loss=mloss(0.8),
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        metrics=[score],
        run_eagerly=False,
        steps_per_execution=64,
    )
    return model


model = build_model()
model.summary()




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ conv_input          │ (None, 40, 60,    │          0 │ -                 │
│ (InputLayer)        │ 60, 1)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d (Conv3D)     │ (None, 38, 58,    │        896 │ conv_input[0][0]  │
│                     │ 58, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_1 (Conv3D)   │ (None, 36, 56,    │     55,360 │ conv3d[0][0]      │
│                     │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 36, 56,    │        256 │ conv3d_1[0][0]    │
│ (BatchNormalizatio… │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d       │ (None, 18, 28,    │          0 │ batch_normalizat… │
│ (MaxPooling3D)      │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_2 (Conv3D)   │ (None, 18, 28,    │    110,656 │ max_pooling3d[0]… │
│                     │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_3 (Conv3D)   │ (None, 18, 28,    │    110,656 │ conv3d_2[0][0]    │
│                     │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 18, 28,    │          0 │ conv3d_3[0][0],   │
│                     │ 28, 64)           │            │ max_pooling3d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_4 (Conv3D)   │ (None, 14, 24,    │  1,024,128 │ add[0][0]         │
│                     │ 24, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_5 (Conv3D)   │ (None, 10, 20,    │  2,048,128 │ conv3d_4[0][0]    │
│                     │ 20, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 20,    │        512 │ conv3d_5[0][0]    │
│ (BatchNormalizatio… │ 20, 128)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_1     │ (None, 5, 10, 10, │          0 │ batch_normalizat… │
│ (MaxPooling3D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_6 (Conv3D)   │ (None, 5, 10, 10, │  2,048,128 │ max_pooling3d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_7 (Conv3D)   │ (None, 5, 10, 10, │  2,048,128 │ conv3d_6[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 5, 10, 10, │          0 │ conv3d_7[0][0],   │
│                     │ 128)              │            │ max_pooling3d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 5, 10, 10, │        512 │ add_1[0][0]       │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_d             │ (None, 5)         │          0 │ -               

 Total params: 7,511,686 (28.65 MB)

 Trainable params: 7,511,046 (28.65 MB)

 Non-trainable params: 640 (2.50 KB)

In [15]:
def get_lr_callback():
    lr_start = 0.001
    lr_min = 0.0001
    lr_ramp_ep = 5
    lr_sus_ep = 0

    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = lr_start - (lr_start - lr_min) / lr_ramp_ep * epoch
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_min
        else:
            lr = lr_min
        return lr

    return tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=False)




In [16]:
# BUGFIX: Create fold split on unique patients (prevents leakage and avoids odd empty-batch artifacts)
unique_patients, unique_inv = np.unique(patient_ids_train, return_inverse=True)

KF = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold0 = next(iter(KF.split(unique_patients)))
t_pidx, v_pidx = fold0

t_mask = np.isin(unique_inv, t_pidx)
v_mask = np.isin(unique_inv, v_pidx)
t_idx = np.where(t_mask)[0]
v_idx = np.where(v_mask)[0]

# Build a tabular-only "fast_model" by feeding a repeated zero CT volume
tab_inp = model.get_layer("input_d").input  # (None,5)

img_dyn = L.Lambda(
    lambda t: tf.repeat(_ZERO_VOL_TF_1, repeats=tf.shape(t)[0], axis=0),
    name="zero_volume_repeat",
)(
    tab_inp
)  # (batch,D,H,W,1)

# BUGFIX: Call the two-input model with two tensors, not a single list interpreted as 1 input.
pred = model(img_dyn, tab_inp)
fast_model = tf.keras.Model(inputs=tab_inp, outputs=pred, name="fast_model")

fast_model.compile(
    loss=model.loss,
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[score],
    run_eagerly=False,
    steps_per_execution=64,
)

tr_ds = tf.data.Dataset.from_tensor_slices((x_train[t_idx], y_train[t_idx]))
tr_ds = tr_ds.shuffle(buffer_size=len(t_idx), seed=SEED, reshuffle_each_iteration=True)
tr_ds = tr_ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

va_ds = tf.data.Dataset.from_tensor_slices((x_train[v_idx], y_train[v_idx]))
va_ds = va_ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(tf.data.AUTOTUNE)

steps_per_epoch = int(np.ceil(len(t_idx) / BATCH_SIZE))
validation_steps = int(np.ceil(len(v_idx) / BATCH_SIZE))

history = fast_model.fit(
    tr_ds,
    epochs=EPOCHS,
    validation_data=va_ds,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    verbose=1,
    callbacks=[get_lr_callback()],
)



ValueError: Layer "functional" expects 2 input(s), but it received 1 input tensors. Inputs received: [<tf.Tensor: shape=(0, 40, 60, 60, 1), dtype=float32, numpy=array([], shape=(0, 40, 60, 60, 1), dtype=float32)>]

In [17]:
te_ds = (
    tf.data.Dataset.from_tensor_slices(x_test)
    .batch(8, drop_remainder=False)
    .prefetch(tf.data.AUTOTUNE)
)

pred = fast_model.predict(
    te_ds,
    verbose=1,
)
pred.shape



NameError: name 'fast_model' is not defined

In [18]:
conf = pred[:, 2] - pred[:, 0]
conf = np.maximum(conf, 70.0)

pred_fvc = pred[:, 1]

pred_df = pd.DataFrame({"FVC": pred_fvc, "Confidence": conf})
pred_df.head()



NameError: name 'pred' is not defined

In [19]:
sub_out = sub.copy()

if len(pred_df) != len(sub_out):
    raise RuntimeError(
        f"Prediction length {len(pred_df)} != submission rows {len(sub_out)}"
    )

sub_out["FVC"] = pred_df["FVC"].values
sub_out["Confidence"] = pred_df["Confidence"].values

subm = sub_out[["Patient_Week", "FVC", "Confidence"]].copy()

subm["FVC"] = subm["FVC"].astype(np.float32).fillna(0)
subm["Confidence"] = subm["Confidence"].astype(np.float32).fillna(70)

subm.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", subm.shape)
print(subm.head(10).to_string(index=False))

NameError: name 'pred_df' is not defined